# Ćwiczenia 1 — Środowisko, dane i kształt p ≫ n (wersja Python, z rozwiązaniami)
**Dane demo: mikrobiom jelitowy w cukrzycy typu 2 (QinJ_2012)**

## 1. Wczytanie kohorty demo

Wersja Python pracuje na danych wyeksportowanych z R do CSV (patrz wersja R, sekcja 2). `X` to próbki × taksony, `y` to etykieta grupy, `meta` to metadane kliniczne. Plik etykiet ma jedną kolumnę `grupa`, którą spłaszczamy do serii.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# X: macierz cech (próbki × taksony), y: etykieta grupy, meta: metadane kliniczne
X = pd.read_csv("dane/t2d_features.csv", index_col=0)
y = pd.read_csv("dane/t2d_labels.csv").squeeze()
meta = pd.read_csv("dane/t2d_metadata.csv", index_col=0)

# klasa referencyjna = control, klasa pozytywna = T2D
y = pd.Series(pd.Categorical(y, categories=["control", "T2D"]))

print(X.shape)
print(y.value_counts())

Dane mają kształt `p ≫ n`: cech (taksonów) jest znacznie więcej niż próbek. To pierwsza i najważniejsza cecha problemu — wraca w każdej kolejnej sekcji kursu.

> **✏️ Zadanie 1 — Kształt danych.** Odczytać kodem (nie „na oko"):
>
> 1. Ile jest próbek, a ile taksonów? Który wymiar jest większy?
> 2. Jaki jest rozkład klas? Czy klasy są zbalansowane?
> 3. Jakiego typu są wartości cech i w jakim mieszczą się zakresie?
> 4. Jaki odsetek macierzy cech to zera? Co to mówi o danych?

In [ ]:
print(X.shape)                                     # (liczba próbek, liczba taksonów)
print(y.value_counts())                            # liczba próbek w każdej klasie
print(y.value_counts(normalize=True).round(3))     # normalize=True: udziały klas zamiast liczności
print(X.dtypes.unique())                           # typ wartości cech (float64 = liczby rzeczywiste)
print(X.to_numpy().min(), X.to_numpy().max())      # najmniejsza i największa wartość w całej macierzy
print((X.to_numpy() == 0).mean())                  # odsetek komórek równych zero

**Rozwiązanie.** Próbek jest 344, taksonów 650, więc cech jest więcej niż próbek (`p > n`).
Klasy są prawie równoliczne (control 174, T2D 170; udziały 0,506 i 0,494). Wartości to liczby
rzeczywiste od 0 do 87,8: liczebności względne w procentach profilu (w Pythonie 11 kolumn
złożonych z samych zer ma typ liczb całkowitych `int64`). Zera stanowią 86,1%
macierzy, czyli większość taksonów w większości próbek nie występuje.

## 2. Zgodność wersji

Od tego spotkania obie wersje operują na tych samych danych. Potwierdzamy, że CSV wczytany w Pythonie zgadza się z tym, co wyeksportowała wersja R.

> **✏️ Zadanie 2 — Zgodność wersji.** Porównać wymiary macierzy i rozkład klas z wartościami odczytanymi w wersji R (`lab_01.Rmd`). Powinny być identyczne — to samo źródło danych.

In [ ]:
print(X.shape)                          # wymiary macierzy cech; w R: dim(X)
print(y.value_counts())                 # rozkład klas; w R: table(y)

**Rozwiązanie.** W obu wersjach macierz ma wymiary 344 × 650, a klasy liczą control 174
i T2D 170. Wyniki są identyczne, bo notatnik Python czyta te same pliki CSV, które
wyeksportowała wersja R.

## 3. Eksploracja cech

Po otwarciu danych warto je najpierw poznać. Wartości w `X` to **liczebności względne** (procenty profilu taksonomicznego), więc suma w każdej próbce jest w przybliżeniu stała. Rozkład pojedynczego taksonu jest zwykle silnie skośny: kilka taksonów dominuje profil, większość występuje śladowo lub wcale.

> **✏️ Zadanie 3 — Dominujące taksony i kompozycyjność.** Zbadać strukturę macierzy cech:
>
> 1. Policzyć sumę każdego wiersza `X` (`X.sum(axis=1)`). Jaką wartość przyjmuje i co z tego wynika o naturze danych?
> 2. Wyznaczyć 5 taksonów o najwyższej średniej liczebności (`X.mean(axis=0)`). Jaki udział profilu zajmuje sam najliczniejszy takson?
> 3. Narysować histogram liczebności najliczniejszego taksonu. Czy rozkład jest symetryczny, czy skośny? (zapowiedź transformacji CLR ze spotkania 3)

In [ ]:
print(X.sum(axis=1).describe())                   # suma każdego wiersza (próbki); axis=1: po wierszach
top5 = X.mean(axis=0).sort_values(ascending=False).head(5)   # średnia każdego taksonu, od największej; pierwsze 5
print(top5.round(2))                              # 5 najliczniejszych taksonów i ich średni udział (%)
print(round(top5.sum(), 1))                       # łączny udział 5 taksonów w profilu (%)
top1 = top5.index[0]                              # nazwa najliczniejszego taksonu
X[top1].plot.hist(bins=30, title=top1)            # histogram jego liczebności, 30 przedziałów
plt.xlabel("liczebność względna (%)")             # podpis osi X
plt.ylabel("liczba próbek")                       # podpis osi Y
plt.show()                                        # wyświetlenie ryciny
print(X[top1].agg(["mean", "median", "max"]).round(2))   # średnia, mediana, maksimum

**Rozwiązanie.** Suma każdego wiersza wynosi prawie dokładnie 100 (od 98,45 do 100,00), więc
dane są kompozycyjne: wartości to udziały w profilu, a wzrost jednego taksonu musi oznaczać
spadek innych. Pięć najliczniejszych taksonów (*Phocaeicola vulgatus*, *Prevotella copri*,
*Bacteroides uniformis*, *B. stercoris*, *Faecalibacterium prausnitzii*) zajmuje łącznie około
35% profilu, a sam najliczniejszy średnio 9,7%. Jego rozkład jest silnie prawostronnie skośny:
średnia (9,71) jest większa od mediany (6,87), a pojedyncze próbki sięgają 66%. Skośność
i kompozycyjność są powodem przekształcenia CLR wprowadzanego na Ćwiczeniach 3.

## 4. Metadane kohorty i zmienne zakłócające

Eksploracja to także pytanie, czy grupy `control` i `T2D` różnią się czymś **poza mikrobiomem**. Jeśli różnią się wiekiem, płcią czy głębokością sekwencjonowania, model może nauczyć się sygnału związanego z tymi zmiennymi, a nie z chorobą. Takie zmienne nazywamy **zakłócającymi** (confounders).

> **✏️ Zadanie 4 — Metadane i zmienne zakłócające.** Na podstawie `meta`:
>
> 1. Które zmienne są stałe w całej kohorcie (ta sama wartość u wszystkich próbek)? Co z nich wynika dla modelu?
> 2. Porównać wiek (`age`) między grupami: mediany i test Manna-Whitneya (`scipy.stats.mannwhitneyu`). Czy grupy są podobne?
> 3. Porównać płeć (`gender`) między grupami: `pd.crosstab` i test chi-kwadrat (`scipy.stats.chi2_contingency`).
> 4. Porównać głębokość sekwencjonowania (`number_reads`) między grupami.
> 5. Która z badanych zmiennych **nie** różni grup istotnie?
> 6. Jednym–dwoma zdaniami: dlaczego różnica wieku i płci między grupami jest problemem przy budowie klasyfikatora na mikrobiomie?

In [ ]:
from scipy.stats import mannwhitneyu, chi2_contingency   # test Manna–Whitneya (Wilcoxona) i chi-kwadrat

g = meta["study_condition"]                        # klasa każdej próbki z metadanych
print(meta.nunique())                              # liczba różnych wartości w każdej kolumnie; 1 = stała
print(meta.groupby(g)["age"].median())             # mediana wieku w każdej klasie
print(mannwhitneyu(meta.age[g == "control"], meta.age[g == "T2D"]).pvalue)   # wartość p dla różnicy wieku
print(pd.crosstab(meta.gender, g))                 # tabela liczności: płeć × klasa
print(chi2_contingency(pd.crosstab(meta.gender, g))[1])   # [1]: wartość p testu chi-kwadrat
print(meta.groupby(g)["number_reads"].median() / 1e6)    # mediana liczby odczytów (mln)
print(mannwhitneyu(meta.number_reads[g == "control"], meta.number_reads[g == "T2D"]).pvalue)   # wartość p
print(meta.groupby(g)["BMI"].median())             # mediana BMI w każdej klasie
print(mannwhitneyu(meta.BMI[g == "control"], meta.BMI[g == "T2D"]).pvalue)   # wartość p dla różnicy BMI

**Rozwiązanie.** Stałe w całej kohorcie są `country` (wszystkie próbki z Chin)
i `non_westernized` (wszędzie „no”); nie niosą żadnej informacji, która mogłaby różnić klasy.
Grupy wyraźnie różnią się wiekiem (mediana 42 wobec 54,5 roku, p ≈ 6·10⁻¹⁴) i płcią (w T2D
więcej mężczyzn: 106 wobec 64 kobiet, p ≈ 0,012), a także liczbą odczytów (mediana 36,3 wobec
41,0 mln, p ≈ 0,004). Grup nie różni istotnie tylko BMI (mediana 23,7 wobec 24,1, p ≈ 0,098).
Skoro chorzy są starsi i częściej są mężczyznami, różnice w mikrobiomie mogą częściowo
odzwierciedlać wiek i płeć, a nie samą cukrzycę; model „rozpoznający T2D” mógłby w rzeczywistości
rozpoznawać starszy wiek.

## 5. Konsekwencje kształtu `p ≫ n` i wersje środowiska

Wersje pakietów zapisujemy od początku — to podstawa reprodukowalności. W wersji Python służy do tego `pip freeze` (z terminala) lub zapis wersji kluczowych bibliotek.

In [ ]:
import sys
print("Python", sys.version.split()[0])
for pkg in (pd, np):
    print(pkg.__name__, pkg.__version__)

> **✏️ Zadanie 5 — Konsekwencje kształtu `p ≫ n`.** Policzyć kodem:
>
> 1. Stosunek liczby cech do liczby próbek (`X.shape[1] / X.shape[0]`).
> 2. Ile taksonów ma zerową lub bliską zeru wariancję (`X.var(axis=0)`).
> 3. Ile taksonów występuje w mniej niż 10% próbek (`(X > 0).mean(axis=0)`).
> 4. Jednym zdaniem: dlaczego regresji logistycznej (właściwej dla binarnej etykiety) nie da się jednoznacznie dopasować na wszystkich cechach naraz, gdy `p > n`.

In [ ]:
print(X.shape[1] / X.shape[0])                # stosunek liczby cech do liczby próbek
print((X.var(axis=0) == 0).sum())             # wariancja każdej kolumny; ile równych zero
print(((X > 0).mean(axis=0) < 0.10).sum())    # ile taksonów występuje w mniej niż 10% próbek

**Rozwiązanie.** Na jedną próbkę przypada około 1,9 cechy. 11 taksonów ma zerową wariancję
(tę samą wartość we wszystkich próbkach), więc nic nie wnosi. 460 z 650 taksonów występuje
w mniej niż 10% próbek. Regresji logistycznej nie da się jednoznacznie dopasować na wszystkich
cechach naraz, bo parametrów jest więcej niż próbek: istnieje nieskończenie wiele zestawów
współczynników, które idealnie pasują do danych, także do szumu.

> **★ Zadanie rozszerzające.** Spróbować dopasować regresję logistyczną wiążącą etykietę ze wszystkimi taksonami jednocześnie. Sprawdzić, dla ilu współczynników procedura nie zwraca wartości, oraz policzyć odsetek trafnych klasyfikacji na tym samym zbiorze, który posłużył do dopasowania modelu. Rozstrzygnąć, czy taki wynik świadczy o jakości modelu, czy wynika z relacji `p > n`.

In [ ]:
from sklearn.linear_model import LogisticRegression   # regresja logistyczna

y01 = (y == "T2D").astype(int).to_numpy()            # klasa T2D jako 1, control jako 0
# C=np.inf: brak kary (jak glm w R); max_iter=5000: zapas iteracji, żeby algorytm doszedł do końca
model = LogisticRegression(C=np.inf, max_iter=5000).fit(X, y01)
print(model.score(X, y01))                           # odsetek trafnych klasyfikacji na tych samych danych
print(np.isnan(model.coef_).sum())                   # liczba współczynników bez wartości (NaN)
print(np.abs(model.coef_).max())                     # największy współczynnik co do wartości bezwzględnej
# rząd macierzy cech z wyrazem wolnym: tyle parametrów da się wyznaczyć jednoznacznie
print(np.linalg.matrix_rank(np.column_stack([np.ones(len(X)), X.to_numpy()])))

**Rozwiązanie.** Model trafia w 100% próbek, ale to skutek `p > n`, a nie jakość modelu.
W R 307 z 651 współczynników (650 cech + wyraz wolny) nie ma wartości (`NA`): przy 344 próbkach
da się jednoznacznie wyznaczyć tylko 344 parametry. W Pythonie scikit-learn zwraca wszystkie
współczynniki bez braków, ale rząd macierzy też wynosi 344, a bardzo duże współczynniki (rzędu
kilkuset) są typowym objawem idealnego rozdzielenia klas. Dopasowanie do danych użytych do
wyznaczenia modelu nic nie mówi o nowych próbkach; dlatego model ocenia się na danych, których
nie widział.